# 👁️ Argos-Action-1.5B: Fine-Tuning Pipeline (QLoRA)
**Author:** Pedro Lucas Reis ([@pedroreoli](https://huggingface.co/pedroreoli))

This notebook performs 4-bit Parameter-Efficient Fine-Tuning (QLoRA) on **Qwen 2.5 1.5B Instruct** using the **Argos Computer Use MCP Dataset** (`pedroreoli/argos-computer-use-mcp`).

> **GPU Recommended:** Free Google Colab T4 or Kaggle 2x T4 (~12-15 minutes runtime).

In [ ]:
!pip install -q "transformers>=4.45.0" "datasets>=3.0.0" "peft>=0.13.0" "trl>=0.11.0" "bitsandbytes>=0.44.0" accelerate huggingface_hub

In [ ]:
import getpass
from huggingface_hub import login
# Enter your Hugging Face Token (Write permission)
hf_token = getpass.getpass('Enter your Hugging Face Write Token: ')
login(token=hf_token)

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
try:
    from transformers import TrainingArguments
except ImportError:
    from trl import SFTConfig as TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_ID = "pedroreoli/argos-computer-use-mcp"
HUB_MODEL_ID = "pedroreoli/argos-action-1.5b"

print("Loading Argos Dataset...")
dataset = load_dataset(DATASET_ID)

print("Configuring QLoRA Quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
def format_prompts(batch):
    formatted = []
    for messages in batch["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        formatted.append(text)
    return {"text": formatted}

formatted_dataset = dataset.map(format_prompts, batched=True)

training_args = TrainingArguments(
    output_dir="./argos-action-1.5b",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit",
    report_to="none",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_ID
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["test"],
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=1536,
    tokenizer=tokenizer,
    args=training_args
)

trainer.train()

In [ ]:
# Push model & tokenizer to Hugging Face Hub
trainer.push_to_hub(commit_message="feat(model): publish Argos-Action-1.5B LoRA policy")
tokenizer.push_to_hub(HUB_MODEL_ID)
print(f"Model published successfully to: https://huggingface.co/{HUB_MODEL_ID}")